# VinRobotics VR M3.1 — Motion Tracking (Mocap Imitation) — Kaggle End-to-End

Train the VR M3.1 humanoid to **imitate the recorded teleop mocap motion** (`huggingface_upload/data/data_clean.csv`, 3348 usable frames @ ~47.9Hz) using mjlab's motion-tracking task, task id **`VR-M3-1-Tracking-Flat`**.

The motion reference file (`data/motion/vr_m3_1_teleop_motion.npz`, ~6MB) is **precomputed and committed to the repo** — cloning the repo is enough, no manual dataset upload needed (unless your repo is private, see below).

## Setup 1 lần
- Settings → **Accelerator: GPU** (T4/P100 both work — see note below), **Internet: ON**.
- **P100 note**: `mujoco_warp`'s dense/tile-based Cholesky solver fails to compile on Pascal (sm_60) GPUs. Cell 1 auto-detects this and switches to `--env.sim.mujoco.jacobian=sparse`, which routes around it. This is a targeted workaround, not independently verified on real P100 hardware in this session — if training still crashes with an LTO/cuSOLVER error after that, switch Settings → Accelerator to **T4 x2** and Run All again.
- Source auto-clones from `GIT_URL` (set below). If your repo is private or Kaggle has no internet: zip the project (including `data/motion/*.npz`) → upload as a Kaggle Dataset → Add Input, and set `GIT_URL = ''`.

## Mỗi lần chạy lại sau khi reset
- **Run All**. To resume training across sessions: download `results_logs.zip` from the Output tab, add it as an Input dataset next session — the Restore cell auto-detects and resumes.

## 0. Config — chỉnh ở đây, các cell sau không cần sửa

In [ ]:
# ================== CONFIG ==================
TASK = 'VR-M3-1-Tracking-Flat'
NUM_ENVS = 1024                # T4 16GB: 1024 safe; reduce to 256-512 if you hit OOM
MAX_ITERS = 3000               # ~12h GPU/session on Kaggle; tune to time remaining
MOTION_FILE = 'data/motion/vr_m3_1_teleop_motion.npz'  # precomputed, committed to repo
GIT_URL = 'https://github.com/huytrao/vinrobotics_mjlab.git'  # '' to use a Dataset zip instead
USE_WANDB = False              # True if you have a W&B account (add secret WANDB_API_KEY)

# --- Optional extra knobs (all have sane defaults in src/tasks/tracking/config/vr_m3_1/rl_cfg.py) ---
LEARNING_RATE = 1.0e-3         # PPO actor/critic learning rate
SAVE_INTERVAL = 500            # checkpoint every N iterations
EPISODE_LENGTH_S = 10.0        # seconds per episode before reset
RECORD_VIDEO_DURING_TRAIN = False  # write a video every VIDEO_INTERVAL steps while training
VIDEO_INTERVAL = 2000
VIDEO_LENGTH = 200

PROJECT_DIR = '/kaggle/working/vinrobotics_mjlab'
print(f'Task={TASK}, envs={NUM_ENVS}, iters={MAX_ITERS}, motion={MOTION_FILE}, lr={LEARNING_RATE}')

## 1. GPU check — must be Turing (T4, sm_75) or newer

In [ ]:
!nvidia-smi

import subprocess

# mujoco_warp's DENSE constraint solver uses a tile-based Cholesky factorization
# (wp.tile_cholesky) that fails to compile via libmathdx/LTO on Pascal (sm_60:
# P100, K80) GPUs — a toolchain limitation, not a config bug. Forcing the MuJoCo
# jacobian to "sparse" (see training cell) routes around that dense/tile solver
# path entirely, so P100 should work too. Best case (T4/A100/L4, sm_75+): leave
# JACOBIAN='auto' for full performance. Fallback if `sparse` still errors on your
# GPU: switch Settings -> Accelerator to T4 x2.
JACOBIAN = 'auto'
try:
    name = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    print('Detected GPU(s):', name)
    for line in name.splitlines():
        gpu_name, cc = [x.strip() for x in line.split(',')]
        major = int(cc.split('.')[0])
        if major < 7:
            JACOBIAN = 'sparse'
            print(
                f"[WARN] GPU '{gpu_name}' has compute capability {cc} (< 7.0, Pascal). "
                "Forcing --env.sim.mujoco.jacobian=sparse to avoid the dense tile-Cholesky "
                "LTO compile failure. If training still crashes, switch Settings -> "
                "Accelerator to T4 x2 (or newer) and Run All again."
            )
    print('JACOBIAN =', JACOBIAN)
except FileNotFoundError:
    print('nvidia-smi not found — no GPU attached? Check Settings -> Accelerator.')

## 2. Lấy source code (git clone, hoặc zip trong /kaggle/input nếu GIT_URL rỗng)

In [ ]:
import glob, os, shutil, stat, subprocess, zipfile

def force_rmtree(path):
    """rmtree that also clears read-only files (git pack files are ro)."""
    def onerr(func, p, exc_info):
        try:
            os.chmod(p, stat.S_IWRITE)
            func(p)
        except OSError:
            pass
    if os.path.exists(path):
        shutil.rmtree(path, onerror=onerr)

def clone_fresh():
    """Clone into a brand-new temp dir, then swap it into PROJECT_DIR.

    Never clones directly into PROJECT_DIR (residue there would make
    `git clone` die with 'destination path already exists') and never
    runs git while cwd is a deleted directory.
    """
    os.chdir('/kaggle/working')
    fresh = '/kaggle/working/_fresh_clone'
    force_rmtree(fresh)
    subprocess.run(['git', 'clone', GIT_URL, fresh], check=True)
    force_rmtree(PROJECT_DIR)
    if os.path.exists(PROJECT_DIR):
        # Couldn't fully clear it (busy/locked file) — overlay the fresh
        # working files on top. Skip .git: copying pack files over a
        # half-deleted repo fails, and training doesn't need the new .git.
        shutil.copytree(fresh, PROJECT_DIR, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('.git'))
        force_rmtree(fresh)
    else:
        os.rename(fresh, PROJECT_DIR)

def fetch_source():
    have_setup = os.path.exists(os.path.join(PROJECT_DIR, 'setup.py'))
    have_motion = os.path.exists(os.path.join(PROJECT_DIR, MOTION_FILE))
    if have_setup and have_motion:
        print('Source + motion file đã có sẵn, bỏ qua clone.')
        return
    if GIT_URL:
        if have_setup:
            # Stale checkout persisted in /kaggle/working from an earlier
            # run that predates data/motion/*.npz — replace it wholesale.
            print('Checkout cũ thiếu motion file -> clone lại bản mới...')
        clone_fresh()
        return
    # No GIT_URL: source comes from an uploaded Kaggle Dataset zip.
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    src_zips = [z for z in zips if 'results_logs' not in os.path.basename(z)]
    assert src_zips, 'Không tìm thấy zip source trong /kaggle/input — upload dataset chứa vinrobotics_mjlab.zip hoặc điền GIT_URL.'
    print('Dùng zip:', src_zips[0])
    tmp = '/kaggle/working/_src'
    force_rmtree(tmp)
    with zipfile.ZipFile(src_zips[0]) as z:
        z.extractall(tmp)
    for root, dirs, files in os.walk(tmp):
        if 'setup.py' in files:
            force_rmtree(PROJECT_DIR)
            shutil.move(root, PROJECT_DIR)
            break
    force_rmtree(tmp)

fetch_source()
assert os.path.exists(os.path.join(PROJECT_DIR, 'setup.py')), 'Lấy source thất bại!'
os.chdir(PROJECT_DIR)
print('OK — project tại', PROJECT_DIR)

motion_path = os.path.join(PROJECT_DIR, MOTION_FILE)
assert os.path.exists(motion_path), (
    f'Motion file khong tim thay tai {motion_path}. '
    f'Kiem tra repo {GIT_URL or "(zip dataset)"} da commit data/motion/*.npz chua '
    '(https://github.com/huytrao/vinrobotics_mjlab/tree/main/data/motion).'
)
print('Motion file OK:', motion_path, f'({os.path.getsize(motion_path)/1e6:.1f} MB)')

## 3. Khôi phục checkpoint từ phiên trước (nếu có `results_logs.zip` trong Input)

In [ ]:
import glob, zipfile, os

RESUME = False
prev = glob.glob('/kaggle/input/**/results_logs.zip', recursive=True)
if prev:
    print('Khôi phục logs từ:', prev[0])
    with zipfile.ZipFile(prev[0]) as z:
        z.extractall(PROJECT_DIR)
    ckpts = glob.glob('logs/rsl_rl/**/model_*.pt', recursive=True)
    if ckpts:
        RESUME = True
        print(f'Đã khôi phục {len(ckpts)} checkpoint → training sẽ RESUME.')
else:
    print('Không có kết quả phiên trước → train từ đầu.')

## 4. Cài dependencies + môi trường headless

In [ ]:
import subprocess, sys, os

# [FIX 1] Nâng kaggle lên phiên bản mới nhất trước
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle'], check=False)

!pip install -q mjlab==1.4.0 mujoco==3.8.1 mujoco-warp==3.8.1 warp-lang==1.13.0 prettytable
!pip install -q -e . --no-deps
!apt-get -qq install -y libegl1 libgl1 libosmesa6 > /dev/null 2>&1 || true

os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

if USE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
else:
    os.environ['WANDB_MODE'] = 'offline'

# [FIX 2] Torch phiên bản mới hơn cho GPU Turing/Ampere (mjlab yêu cầu torch>=2.7.0;
# nếu bản cu121 dưới đây thấp hơn, resolver sẽ nâng lên bản phù hợp khi cần).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu121'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow'], check=False)

import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    x = torch.randn(512, 512, device='cuda')
    print('GPU test OK:', float((x @ x).sum()))

## 5. Liệt kê task (`scripts/list_envs.py`)

In [ ]:
!python scripts/list_envs.py

## 5b. (Optional) Regenerate the motion file from a different mocap CSV
Skip this cell if `data/motion/vr_m3_1_teleop_motion.npz` was already restored/cloned — it's precomputed. Only run this if you want to rebuild it from a fresh `data_clean.csv` (e.g. new teleop recording, via `scripts/clean_mocap_data.py`).

In [ ]:
REGENERATE_MOTION = False  # set True to rebuild from huggingface_upload/data/data_clean.csv
if REGENERATE_MOTION:
    !python scripts/mocap_csv_to_motion_npz.py \
        --csv-path huggingface_upload/data/data_clean.csv \
        --output-path {MOTION_FILE} \
        --device cpu
else:
    print('Skipped — using precomputed', MOTION_FILE)

## 6. Training (`scripts/train.py`)
Tự resume nếu cell 3 đã khôi phục checkpoint. Checkpoint + `policy.onnx` lưu tại `logs/rsl_rl/<exp>/<datetime>/`.

In [ ]:
resume_flag = '--agent.resume True' if RESUME else ''
video_flags = (
    f'--video True --video-interval {VIDEO_INTERVAL} --video-length {VIDEO_LENGTH}'
    if RECORD_VIDEO_DURING_TRAIN else ''
)
!python scripts/train.py {TASK} \
    --env.scene.num-envs={NUM_ENVS} \
    --env.sim.mujoco.jacobian={JACOBIAN} \
    --env.episode-length-s={EPISODE_LENGTH_S} \
    --agent.max-iterations={MAX_ITERS} \
    --agent.algorithm.learning-rate={LEARNING_RATE} \
    --agent.save-interval={SAVE_INTERVAL} \
    --motion-file {MOTION_FILE} \
    --gpu-ids '[0]' {resume_flag} {video_flags}

## 7. Play + render video (`scripts/play.py`)
Kaggle has no DISPLAY -> use `--video` to export mp4. `play.py` takes a specific
checkpoint **file** via `--checkpoint-file` (not a directory).

In [ ]:
import glob, os

ckpts = glob.glob('logs/rsl_rl/**/model_*.pt', recursive=True)
ckpts.sort(key=os.path.getmtime)

if not ckpts:
    raise SystemExit('KHÔNG có checkpoint nào — chạy lại cell Training (mục 6) và kiểm tra output.')

CKPT = ckpts[-1]
print('Checkpoint:', CKPT)

!timeout 300 python scripts/play.py {TASK} \
    --checkpoint-file "{CKPT}" \
    --motion-file {MOTION_FILE} \
    --video --video-length 200 || true

## 8. Batch export ONNX

`scripts/export_policy.py` has no `--motion-file` flag, but tracking tasks
build their `MotionCommand` (and load the motion npz) eagerly at environment
construction, so it can't build the env at all for this task without one.
We replicate its export loop inline here, setting `motion_file` on the loaded
env config before creating the environment.

In [ ]:
import glob, os, re
from dataclasses import asdict
from pathlib import Path

import torch
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.tasks.tracking.mdp import MotionCommandCfg
import mjlab.tasks, src.tasks  # noqa: F401  (populate the task registry)

ckpts = glob.glob('logs/rsl_rl/**/model_*.pt', recursive=True)
ckpts.sort(key=os.path.getmtime)
if not ckpts:
    raise SystemExit('KHÔNG có checkpoint nào trong logs/rsl_rl/ — chạy lại mục 6.')

CKPT = ckpts[-1]
checkpoint_dir = Path(os.path.dirname(CKPT))
last_iter = int(re.search(r'model_(\d+)', os.path.basename(CKPT)).group(1))
print('Checkpoint dir:', checkpoint_dir, '| latest iter:', last_iter)

EXPORT_START, EXPORT_END, EXPORT_STEP = 0, last_iter, 500

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
env_cfg = load_env_cfg(TASK, play=True)
agent_cfg = load_rl_cfg(TASK)
env_cfg.scene.num_envs = 1

motion_cmd = env_cfg.commands['motion']
assert isinstance(motion_cmd, MotionCommandCfg)
motion_cmd.motion_file = MOTION_FILE

env = ManagerBasedRlEnv(cfg=env_cfg, device=device)
env = RslRlVecEnvWrapper(env, clip_actions=agent_cfg.clip_actions)

runner_cls = load_runner_cls(TASK) or MjlabOnPolicyRunner
runner = runner_cls(env, asdict(agent_cfg), device=device)

export_dir = checkpoint_dir / 'exported'
os.makedirs(export_dir, exist_ok=True)

exported = 0
for i in range(EXPORT_START, EXPORT_END + 1, EXPORT_STEP):
    cp = checkpoint_dir / f'model_{i}.pt'
    if not cp.exists():
        continue
    print(f'[INFO] Exporting {cp.name} ...')
    runner.load(str(cp), load_cfg={'actor': True}, strict=True, map_location=device)
    runner.export_policy_to_onnx(str(export_dir), filename=f'policy_{i}.onnx')
    exported += 1

env.close()
print(f'[INFO] Done. Exported {exported} ONNX models to {export_dir}')

In [ ]:
!rm -f /kaggle/working/results_logs.zip
!zip -qr /kaggle/working/results_logs.zip logs -x '*wandb*'
!ls -lh /kaggle/working/results_logs.zip
print('\nXong! Tải results_logs.zip từ tab Output, phiên sau add nó làm Input để tiếp tục train.')